In [ ]:
link = "https://www.imdb.com/pt/chart/top/"

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import json
import os
import requests


service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

In [ ]:
driver.get(link)

In [ ]:
espera = WebDriverWait(driver, 15)

In [ ]:
espera.until(EC.presence_of_element_located((By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item a")))
links = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item a")
lista_de_dados = []

for link in links:
    texto = link.text.strip() # O .strip() remove espaços em branco acidentais
    
    # Se o texto NÃO for vazio, nós pegamos a URL e imprimimos
    if texto: 
        url = link.get_attribute("href")
        
        dicionario = dict()
        dicionario["titulo"] = texto
        dicionario["url"] = url
        lista_de_dados.append(dicionario)
        
        print(f"Texto: {texto} | URL: {url}")

In [ ]:
len(lista_de_dados)

In [ ]:
for filme in lista_de_dados:
    driver.get(filme["url"])
    
    # Espera e pega pela url do poster
    a = espera.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a.ipc-lockup-overlay.ipc-focusable.ipc-focusable--constrained")))
    filme["url_imagem"] = a.get_attribute("href")
    if not filme["url_imagem"]:
        print(f"Não foi encontrado url do poster do filme {filme['titulo']}")
        
    # Pega ano de lançamento e a duração
    ano_idade_duracao = driver.find_elements(By.CSS_SELECTOR, "ul.ipc-inline-list.ipc-inline-list--show-dividers.sc-b41e510f-3.ggypaO.baseAlt li")
    if len(ano_idade_duracao) >= 3:
        filme["ano"] = ano_idade_duracao[0].text
        filme["duracao"] = ano_idade_duracao[2].text
    elif len(ano_idade_duracao) == 2:
        filme["ano"] = ano_idade_duracao[0].text
        filme["duracao"] = ano_idade_duracao[1].text
    else:
        filme["ano"] = None
        filme["duracao"] = None
        print(f"Erro ao pegar idade e ano do filme: {filme['titulo']} -- {filme['url']}")
    
    # Lista de Generos
    generos = driver.find_elements(By.CSS_SELECTOR, "div.ipc-chip-list__scroller a")
    filme["generos"] = []
    for genero in generos:
        filme["generos"].append(genero.text)

    # Pegar diretores
    try:
        # O contains(@href, '/name/') garante que pegaremos links de pessoas
        xpath_diretores = "(//li[@data-testid='title-pc-principal-credit'])[1]//a[contains(@href, '/name/')]"
        elementos_diretores = espera.until(EC.presence_of_all_elements_located((By.XPATH, xpath_diretores)))
        filme["diretores"] = [d.get_attribute("textContent").strip() for d in elementos_diretores]
    except Exception as e:
        print(f"Não foi possível extrair o diretor. Erro: {e}")
        filme["diretores"] = None
        
    elementos_nota = espera.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "[data-testid='hero-rating-bar__aggregate-rating__score'] span")))

    # Pegando o primeiro não vazio, porque geralmente há algum vazio antes
    for elemento in elementos_nota:
        texto_extraido = elemento.get_attribute("textContent").strip()
        if texto_extraido != "":
            nota_imdb = texto_extraido
            filme["Nota"] = nota_imdb
            break 


    print(f"Pegamos do filme {filme}")

In [ ]:
lista_de_dados

In [ ]:
def validar_filmes(lista_filmes):
    # campos obrigatórios em cada json
    campos_obrigatorios = ["titulo", "url", "url_imagem", "ano", "duracao", "generos", "diretores", "Nota"]
    erros_encontrados = []

    for index, filme in enumerate(lista_filmes):
        erros_do_item = []
        
        # Verificar se o título existe para identificar o filme no log de erro
        nome_filme = filme.get("titulo", f"Filme sem título (Posição {index})")

        for campo in campos_obrigatorios:
            if campo not in filme:
                erros_do_item.append(f"Campo ausente: '{campo}'")
            else:
                valor = filme[campo]
                
                # Verifica se o valor é vazio (string vazia, lista vazia ou None)
                if valor is None or valor == "" or (isinstance(valor, list) and len(valor) == 0):
                    erros_do_item.append(f"Campo vazio ou inválido: '{campo}'")

        # Se houver erros neste filme, adicionamos ao relatório
        if erros_do_item:
            erros_encontrados.append({
                "filme": nome_filme,
                "erros": erros_do_item
            })

    return erros_encontrados

relatorio_erros = validar_filmes(lista_de_dados)

if not relatorio_erros:
    print("Nenhum erro encontrado! Todos os dados estão completos.")
else:
    print(f"Foram encontrados problemas em {len(relatorio_erros)} itens:\n")
    for item in relatorio_erros:
        print(f"Filme: {item['filme']}")
        for erro in item['erros']:
            print(f"   - {erro}")
        print("-" * 30)

In [ ]:
# Havia esquecido de fazer o doawload das imagens, e nesse processo percebi que minhas url_images não eram de fato url das imagens, e sim de uma visualização do IMDb das imagens, essa celula interia trata de atualizar as urls para urls de imagens de fato, e então baixar elas 

def limpar_nome_arquivo(titulo):
    # Converte para minúsculo, remove acentos simples e substitui espaços
    nome_limpo = titulo.lower().strip()
    replacements = [(" ", "_"), (":", ""), ("á", "a"), ("é", "e"), ("í", "i"), ("ó", "o"), ("ú", "u"), ("ñ", "n"), ("ç", "c")]
    for velho, novo in replacements:
        nome_limpo = nome_limpo.replace(velho, novo)
    return f"{nome_limpo}.png"

def extrair_jpg_com_selenium(url_pagina):
    """Usa o Selenium para carregar o JS e extrair a tag <img> do media viewer (o link que eu salvei por engano)."""
    try:
        driver.get(url_pagina)
        
        # Espera até 10 segundos no máximo para carregar o link
        elemento_imagem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//div[@data-testid="media-viewer"]//img'))
        )
        
        # Pega o atributo 'src' (que é o link direto do .jpg)
        link_jpg = elemento_imagem.get_attribute("src")
        return link_jpg
        
    except Exception as e:
        print(f"Erro ao extrair com Selenium na URL {url_pagina}: {e}")
        return None


print("\nExtraindo os links diretos (.jpg) com Selenium...")
for filme in lista_de_dados:
    print(f"Buscando link para: {filme['titulo']}")
    link_direto = extrair_jpg_com_selenium(filme['url_imagem'])
    
    if link_direto:
        filme['url_imagem'] = link_direto
        print(f"   -> Encontrado: {link_direto}")
    else:
        print("   -> Link não encontrado.")

# Fecha o navegador pois não precisamos mais dele
driver.quit()
print("Navegador fechado.\n")


print("Baixando as imagens com Requests...")
if not os.path.exists('posters'):
    os.makedirs('posters')

for filme in lista_de_dados:
    if 'url_imagem' in filme:
        titulo = filme['titulo']
        url_jpg = filme['url_imagem']
        
        nome_limpo = limpar_nome_arquivo(titulo)
        caminho_arquivo = f"posters/{nome_limpo}.jpg"
        
        try:
            resposta = requests.get(url_jpg, headers={'User-Agent': 'Mozilla/5.0'}, stream=True)
            if resposta.status_code == 200:
                with open(caminho_arquivo, 'wb') as f:
                    for chunk in resposta.iter_content(1024):
                        f.write(chunk)
                print(f"Salvo: {caminho_arquivo}")
            else:
                print(f"Erro ao baixar {titulo} (Status: {resposta.status_code})")
        except Exception as e:
            print(f"Falha no download de {titulo}: {e}")

In [ ]:
# salvar em arquivo json
with open("dados.json", "w", encoding="utf-8") as f:
    json.dump(lista_de_dados, f, ensure_ascii=False, indent=4)